# Lesson 000.08 -- Tree Branches from Brackets: AST Tree Structures

AuthorityEffect: None.

> **Claim-typing legend**
> - **[Pinned]**: reproduced from verified checked-in Grammar 000 specifications.
> - **[Executed]**: verified by deterministic Python execution in this notebook.
> - **[Declared]**: pedagogical convention or beginner framing.
> - **[Open]**: questions left for future exploration.

## 1. The Big Idea: Nested Packets as Ordered Syntax Trees

[Declared] In Grammar 000, application boxes can contain other applications as children. This recursive nesting creates an **Ordered Abstract Syntax Tree (AST)**:
- **Leaf nodes**:
  - **Structural leaves**: Any terminal node with no children: `Diff()`, `Atom(word)`, and childless `App(())` (`op<>`).
  - **Data leaves**: Terminal nodes carrying payload or ground distinction: `Diff()` and `Atom(word)`.
- **Branch nodes**: `App(children)` nodes holding an ordered tuple of subtrees.

Consider the nested wire string:
```text
op<<0>op<<1><>>>
```
This produces an AST with:
- Root: `App`
  - Child 0: `Atom("0")` (leaf)
  - Child 1: `App` (branch)
    - Sub-child 0: `Atom("1")` (leaf)
    - Sub-child 1: `Diff()` (leaf)

## 2. Tree Traversal, Depth, and Leaves

[Executed] We can analyze the structural properties of any packet tree:
- **Tree Depth (App Nesting Depth)**: The maximum nesting depth of `App` nodes (depth 0 for an Atom or Diff; depth 1 for childless `App(())`; depth 2 for `op<<0>op<<1><>>>`).
- **Leaf Count**: Total number of structural leaves (including empty `App(())`) or data leaves.
- **Node Count**: The total number of nodes (branches + leaves).


In [ ]:
# [Pinned] Environment Preflight & Project Root Resolution
# Pinned revision: 3f3399766108adc4c4e7615dce2573a57be57dd6
# Supported environment: Python 3.12+ (e.g. py -3.12 or pixi run)
# Project install: pip install -e .
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().resolve()
found = False
while ROOT_DIR.parent != ROOT_DIR:
    if (
        (ROOT_DIR / "pfp").exists()
        or (ROOT_DIR / ".git").exists()
        or (ROOT_DIR / "pixi.toml").exists()
    ):
        if (ROOT_DIR / "pfp").exists():
            found = True
            break
    ROOT_DIR = ROOT_DIR.parent

if found and str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))
elif not found:
    raise RuntimeError(
        "Could not locate 'pfp' root. "
        "Ensure you execute from a repository checkout or install via: pip install -e ."
    )

from pfp.domain.tower import grammar

packet = grammar.parse("op<<0>op<<1><>>>")


def collect_leaves(p, structural=True):
    """Collects leaves. If structural=True, childless App(()) is included as a leaf."""
    if not hasattr(p, "children") or not p.children:
        if structural or isinstance(p, (grammar.Diff, grammar.Atom)):
            return [p]
        return []
    leaves = []
    for c in p.children:
        leaves.extend(collect_leaves(c, structural=structural))
    return leaves


def tree_depth(p):
    """Returns App nesting depth: 0 for Diff/Atom, 1 for childless App, 1+max for nested App."""
    if not isinstance(p, grammar.App):
        return 0
    if not p.children:
        return 1
    return 1 + max(tree_depth(c) for c in p.children)


# 1. Structural verification of AST
assert isinstance(packet, grammar.App)
assert len(packet.children) == 2
assert isinstance(packet.children[0], grammar.Atom) and packet.children[0].word == "0"

sub = packet.children[1]
assert isinstance(sub, grammar.App)
assert len(sub.children) == 2
assert isinstance(sub.children[0], grammar.Atom) and sub.children[0].word == "1"
assert isinstance(sub.children[1], grammar.Diff)

# 2. Leaf collection in left-to-right order
leaves = collect_leaves(packet)
assert len(leaves) == 3
assert isinstance(leaves[0], grammar.Atom) and leaves[0].word == "0"
assert isinstance(leaves[1], grammar.Atom) and leaves[1].word == "1"
assert isinstance(leaves[2], grammar.Diff)

# 3. Depth calculation on nested and empty applications
assert tree_depth(packet) == 2
empty_app = grammar.parse("op<>")
assert tree_depth(empty_app) == 1  # Empty App has nesting depth 1
assert len(collect_leaves(empty_app, structural=True)) == 1  # 1 structural leaf (childless App)
assert len(collect_leaves(empty_app, structural=False)) == 0  # 0 data leaves

nested_empty = grammar.parse("op<op<>>")
assert tree_depth(nested_empty) == 2

print("Tree structure verified: 3 leaves, depth 2, and empty-App conventions confirmed.")

## 3. Diagnostic Probe (2026 Micro-Attention Check)

**Question:** In the packet `op<<0>op<<1><>>>`, how many total leaves (terminal nodes) exist?

- **A:** 2 leaves: `<0>` and `<1>`.
- **B:** 3 leaves: `<0>`, `<1>`, and `<>`.
- **C:** 4 leaves.

*(Correct answer: **B**. `Diff()` is an actual terminal node of the AST, so the leaves are `<0>`, `<1>`, and `<>`.)*

In [ ]:
# Interactive Diagnostic Probe Check
student_choice = "B"  # Change to experiment
assert student_choice == "B", "Incorrect! Expected B: 3 leaves (<0>, <1>, and <>)"

# Mechanically verify the property
leaves = collect_leaves(grammar.parse("op<<0>op<<1><>>>"))
assert len(leaves) == 3
assert [isinstance(l, grammar.Diff) for l in leaves] == [False, False, True]

print("Diagnostic Probe: PASSED (3 terminal leaves verified)")

## 4. Student Exercises (Ages 5 to 500)

1. **Exercise 1 (Age 5):** Look at a tree with two main branches. Each branch has twigs. Can you draw a tree for `op<<branch1><branch2>>`?
2. **Exercise 2 (Age 15):** Parse `op<op<<0>>op<<1>>>`. What is its depth, and what are its children? Test it below!
3. **Exercise 3 (Age 500):** Write a recursive function `collect_leaves(packet)` and verify that it extracts all terminal nodes in strict left-to-right order.

In [ ]:
# Exercise Solution & Verification
p_nested = grammar.parse("op<op<<0>>op<<1>>>")
assert isinstance(p_nested, grammar.App)
assert len(p_nested.children) == 2
assert tree_depth(p_nested) == 2

leaves_nested = collect_leaves(p_nested)
assert len(leaves_nested) == 2
assert [l.word for l in leaves_nested] == ["0", "1"]

print("Student exercises verified successfully: nested tree depth and leaf extraction.")

## 5. Anchors & References

| Anchor Type | Target Link |
| :--- | :--- |
| **Curriculum Anchor** | [Grammar 000 AST](../../../../symbol_game/domain/tower/grammar.py) |
| **Authority Effect** | `None` |
| **AST Invariant** | Deterministic ordered tree traversal and recursive decomposition |